# Elo-adjusted work-mode sequence-slope model — combined population

This notebook replaces the first-half/later-half progress score with an attempt-level trajectory model. It uses every retained first attempt in each eligible activity sequence and estimates one adjusted work-mode difference in trajectory slopes.

The primary model is:

`success ~ centered_work_mode * centered_position + centered_exercise_elo`  
`        + (1 | classroom_id) + (1 | student_id)`  
`        + (1 + centered_position | sequence_id)`

`normalized_position` runs from 0 at the first exercise to 1 at the last exercise. The fitted variable is `centered_position = normalized_position - 0.5`, so the start is -0.5, the midpoint is 0, and the end is +0.5. The ZPDES indicator is also centered at its modeled-row mean to reduce intercept–mode covariance; because playlist and ZPDES still differ by exactly one on that centered contrast, `zpdes_x_centered_position` remains the complete ZPDES-minus-playlist slope difference. GPBoost fits a Bernoulli-logit model and represents the sequence intercept and slope as independent variance components.

The primary population is `combined`, matching the reported half–half model: it includes students observed in only one work mode as well as students observed in both. This makes the samples comparable across the two trajectory definitions, but it is a system-level observational comparison rather than a purely within-student contrast. The activity sequences used here are the same activity/playlist-qualified sequences as the progress notebook; they are not sessionized using a time-gap rule.

## 1. Setup

Reusable data construction and model-fitting functions live in `scripts/model_work_mode_sequence_slope.py`.

In [1]:
from __future__ import annotations

import gc
import importlib
import sys
from argparse import Namespace
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import scripts.model_work_mode_sequence_slope as sequence_slope_model  # noqa: E402

importlib.reload(sequence_slope_model)

from scripts.model_work_mode_progress import load_attempts, split_populations  # noqa: E402
from scripts.model_work_mode_sequence_slope import (  # noqa: E402
    build_sequence_slope_trajectory,
    fit_sequence_slope_model,
)

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 140)

## 2. Parameters

`MODEL_POPULATION = 'combined'` matches the population used by the reported half–half model. It includes both exclusive-mode and both-mode students. Use `'both_modes'` only for a separate within-student-population sensitivity analysis.

Four exercises is the minimum inherited from the progress notebook. Sequence random slopes based on four observations remain noisy individually, but the hierarchical model partially pools their common variance instead of estimating and averaging a separate unpooled regression for every sequence.

In [2]:
INPUT_FILE = PROJECT_ROOT / 'data_MIA' / '986-neurips-mia_20260415_100024.parquet'
EXERCISE_CATALOG_JSON = PROJECT_ROOT / 'data_MIA' / 'exo_mia.json'
MODULE_CONFIG_JSON = PROJECT_ROOT / 'data_MIA' / 'config_mia.json'
EXERCISE_ELO_FILE = (
    PROJECT_ROOT
    / 'artifacts'
    / 'sources'
    / 'mia'
    / 'artifacts'
    / 'derived'
    / 'agg_exercise_elo.parquet'
)

MODEL_POPULATION = 'combined'
MIN_SEQUENCE_EXERCISES = 4
POSITION_BINS = 20
MAXITER = 200
RUN_MODEL = True
OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'work_mode_sequence_slope_combined_notebook'

if MODEL_POPULATION not in {'exclusive_modes', 'both_modes', 'combined'}:
    raise ValueError('MODEL_POPULATION must be exclusive_modes, both_modes, or combined')
for required_path in (
    INPUT_FILE, EXERCISE_CATALOG_JSON, MODULE_CONFIG_JSON, EXERCISE_ELO_FILE
):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

args = Namespace(
    input_file=INPUT_FILE,
    input_csv=None,
    data_dir=None,
    exercise_catalog_json=EXERCISE_CATALOG_JSON,
    module_config_json=MODULE_CONFIG_JSON,
    keep_only_single_module_playlists=False,
)

## 3. Load the selected population

The loader retains playlist and ZPDES attempts, standardizes identifiers, maps playlist exercises to catalogue activities, and converts correctness to a binary `success` value.

In [3]:
attempts = load_attempts(args)
if MODEL_POPULATION == 'combined':
    selected_attempts = attempts
else:
    population_frames = split_populations(attempts)
    selected_attempts = population_frames[MODEL_POPULATION]
    del population_frames

attempt_summary = pd.DataFrame(
    [
        {
            'population': MODEL_POPULATION,
            'attempt_rows': len(selected_attempts),
            'students': selected_attempts['student_id'].nunique(),
            'classrooms': selected_attempts['classroom_id'].nunique(),
            'modules': selected_attempts['module'].nunique(),
            'playlist_rows': int(selected_attempts['work_mode'].eq('playlist').sum()),
            'zpdes_rows': int(selected_attempts['work_mode'].eq('zpdes').sum()),
        }
    ]
)
display(attempt_summary)

exercise_elo = pd.read_parquet(
    EXERCISE_ELO_FILE,
    columns=['exercise_id', 'activity_id', 'exercise_elo', 'calibrated'],
)
del attempts
gc.collect()

,population,attempt_rows,students,classrooms,modules,playlist_rows,zpdes_rows
0,combined,5590740,37894,3091,27,1633559,3957181


61

## 4. Build eligible normalized-position sequences

The construction matches `work_mode_progress_random_module_model.ipynb`: playlist activities are qualified by playlist ID, only the earliest retained attempt for each student-exercise pair is used, and sequences with fewer than `MIN_SEQUENCE_EXERCISES` are removed. Every remaining sequence receives its own globally unique `sequence_id`.

In [4]:
trajectory = build_sequence_slope_trajectory(
    selected_attempts,
    exercise_elo,
    min_sequence_exercises=MIN_SEQUENCE_EXERCISES,
)
del selected_attempts
gc.collect()

sequence_table = (
    trajectory.groupby(['sequence_id', 'work_mode'], as_index=False, observed=True)
    .agg(
        sequence_exercises=('exercise_id', 'size'),
        student_id=('student_id', 'first'),
        classroom_id=('classroom_id', 'first'),
        module=('module', 'first'),
        activity_id=('activity_id', 'first'),
    )
)
sequence_summary = (
    sequence_table.groupby('work_mode', as_index=False, observed=True)
    .agg(
        sequences=('sequence_id', 'size'),
        students=('student_id', 'nunique'),
        classrooms=('classroom_id', 'nunique'),
        modules=('module', 'nunique'),
        median_exercises=('sequence_exercises', 'median'),
        mean_exercises=('sequence_exercises', 'mean'),
        maximum_exercises=('sequence_exercises', 'max'),
    )
)
elo_coverage = (
    trajectory.groupby('work_mode', as_index=False, observed=True)
    .agg(
        attempt_rows=('success', 'size'),
        elo_rows=('exercise_elo', 'count'),
        mean_exercise_elo=('exercise_elo', 'mean'),
    )
)
elo_coverage['elo_coverage'] = elo_coverage['elo_rows'] / elo_coverage['attempt_rows']
display(sequence_summary.round(2))
display(elo_coverage.round(4))

,work_mode,sequences,students,classrooms,modules,median_exercises,mean_exercises,maximum_exercises
0,playlist,118104,13667,1076,24,10.0,11.06,332
1,zpdes,349909,25489,2501,27,6.0,7.07,71


,work_mode,attempt_rows,elo_rows,mean_exercise_elo,elo_coverage
0,playlist,1306573,1306573,1475.4221,1.0
1,zpdes,2472741,2472741,1541.9384,1.0


## 5. Inspect the observed trajectories

The plot is descriptive. It bins normalized position only for visualization; the regression below uses the continuous, unbinned position of every attempt. The dashed Elo curves show whether exercise difficulty changes over a sequence.

In [5]:
position_bin_index = np.minimum(
    (trajectory['normalized_position'].astype(float) * POSITION_BINS).astype(int),
    POSITION_BINS - 1,
)
plot_rows = trajectory.assign(position_bin_index=position_bin_index)
binned_trajectory = (
    plot_rows.groupby(['work_mode', 'position_bin_index'], as_index=False, observed=True)
    .agg(
        success_rate=('success', 'mean'),
        mean_exercise_elo=('exercise_elo', 'mean'),
        attempt_rows=('success', 'size'),
        sequences=('sequence_id', 'nunique'),
        students=('student_id', 'nunique'),
    )
)
binned_trajectory['normalized_position'] = (
    binned_trajectory['position_bin_index'] + 0.5
) / POSITION_BINS
del plot_rows, position_bin_index

colors = {'playlist': '#DD8452', 'zpdes': '#4C72B0'}
trajectory_figure = make_subplots(specs=[[{'secondary_y': True}]])
for work_mode in ('playlist', 'zpdes'):
    mode_rows = binned_trajectory[binned_trajectory['work_mode'].eq(work_mode)]
    label = 'ZPDES' if work_mode == 'zpdes' else 'Playlist'
    trajectory_figure.add_trace(
        go.Scatter(
            x=mode_rows['normalized_position'],
            y=mode_rows['success_rate'],
            mode='lines+markers',
            name=f'{label}: observed success',
            line={'color': colors[work_mode], 'width': 3},
            customdata=mode_rows[['attempt_rows', 'sequences', 'students']].to_numpy(),
            hovertemplate=(
                'Position: %{x:.2f}<br>Success: %{y:.1%}'
                '<br>Attempts: %{customdata[0]:,.0f}'
                '<br>Sequences: %{customdata[1]:,.0f}'
                '<br>Students: %{customdata[2]:,.0f}<extra></extra>'
            ),
        ),
        secondary_y=False,
    )
    trajectory_figure.add_trace(
        go.Scatter(
            x=mode_rows['normalized_position'],
            y=mode_rows['mean_exercise_elo'],
            mode='lines',
            name=f'{label}: exercise Elo',
            line={'color': colors[work_mode], 'width': 2, 'dash': 'dash'},
            hovertemplate='Position: %{x:.2f}<br>Mean Elo: %{y:.1f}<extra></extra>',
        ),
        secondary_y=True,
    )

trajectory_figure.update_layout(
    title='Observed success and exercise difficulty over normalized sequence position',
    template='simple_white',
    hovermode='x unified',
    legend={'orientation': 'h', 'y': 1.12, 'x': 0.5, 'xanchor': 'center'},
    margin={'l': 70, 'r': 70, 't': 110, 'b': 65},
)
trajectory_figure.update_xaxes(title_text='Normalized sequence position (0 = start, 1 = end)')
trajectory_figure.update_yaxes(
    title_text='Observed first-attempt success', tickformat='.0%', secondary_y=False
)
trajectory_figure.update_yaxes(
    title_text='Mean exercise Elo (higher = harder)', secondary_y=True
)
trajectory_figure.show(config={'displaylogo': False, 'responsive': True})

## 6. Fit the mixed sequence-slope model

The model uses a Bernoulli-logit likelihood. Position is centered at the sequence midpoint, and the ZPDES indicator is centered at its modeled-row mean, to improve numerical conditioning without changing the playlist-versus-ZPDES contrasts. Exercise Elo is centered at the mean Elo of modeled attempts and divided by 100, so its coefficient is the log-odds change associated with a 100-point increase in difficulty.

The adjusted start/end probabilities below hold Elo at that mean and set random effects to zero. They provide an interpretable probability-scale point estimate. `slope_comparison_reportable` requires convergence and finite standard errors for the position, interaction, and Elo terms. `probability_levels_reportable` is stricter: it additionally requires finite standard errors for all baseline terms. Formal uncertainty for the principal comparison is reported for the fixed interaction coefficient on the log-odds scale.

In [6]:
fit_summary = pd.DataFrame()
fixed_effects = pd.DataFrame()
variance_components = pd.DataFrame()
adjusted_changes = pd.DataFrame()
teacher_summary = pd.DataFrame()

if RUN_MODEL:
    fit_result = fit_sequence_slope_model(
        trajectory,
        population=MODEL_POPULATION,
        maxiter=MAXITER,
    )
    fit_summary = pd.DataFrame([fit_result.summary])
    fixed_effects = fit_result.fixed_effects
    variance_components = fit_result.variance_components
    adjusted_changes = fit_result.adjusted_changes

    if not fit_summary.empty:
        teacher_summary = fit_summary[
            [
                'population', 'status', 'converged', 'reportable',
                'slope_comparison_reportable',
                'probability_levels_reportable',
                'n_rows', 'n_sequences', 'n_students', 'n_classrooms',
                'playlist_adjusted_change_points',
                'zpdes_adjusted_change_points',
                'adjusted_difference_in_change_points',
                'slope_difference_log_odds',
                'slope_difference_ci_low', 'slope_difference_ci_high',
                'slope_difference_p_value',
                'invalid_standard_error_terms',
            ]
        ].copy()
        display(teacher_summary.round(3))
        if not bool(teacher_summary.iloc[0]['slope_comparison_reportable']):
            print('Do not report the slope comparison until convergence and its standard errors are valid.')
        elif not bool(teacher_summary.iloc[0]['probability_levels_reportable']):
            print('The slope comparison is reportable. Treat adjusted start/end probabilities as point estimates because baseline-level standard errors are incomplete.')
else:
    print('RUN_MODEL is False; data preparation and descriptive plotting are complete.')

C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning:

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html



,population,status,converged,reportable,slope_comparison_reportable,probability_levels_reportable,n_rows,n_sequences,n_students,n_classrooms,playlist_adjusted_change_points,zpdes_adjusted_change_points,adjusted_difference_in_change_points,slope_difference_log_odds,slope_difference_ci_low,slope_difference_ci_high,slope_difference_p_value,invalid_standard_error_terms
0,combined,ok,True,True,True,False,3779314,468013,34224,2910,6.857,34.77,27.913,1.339,1.319,1.358,0.0,"Intercept, zpdes"


The slope comparison is reportable. Treat adjusted start/end probabilities as point estimates because baseline-level standard errors are incomplete.


## 7. Read the output

The main inferential output is `slope_difference_log_odds`, the coefficient of `zpdes_x_centered_position`. Positive values indicate a more positive start-to-end ZPDES trajectory after adjustment for exercise Elo and the modeled classroom, student, and sequence dependence. Centering work mode changes the intercept and common position coefficient, but not this difference: playlist and ZPDES remain one unit apart. Because start and end are -0.5 and +0.5, their distance also remains one.

`adjusted_difference_in_change_points` is the corresponding fixed-effect probability-scale point contrast at mean exercise Elo. Because the logit link is nonlinear, it is a derived contrast rather than the regression coefficient itself.

In [7]:
if not adjusted_changes.empty:
    probability_summary = adjusted_changes.copy()
    probability_summary['adjusted_start_percent'] = (
        probability_summary['adjusted_start_probability'] * 100
    )
    probability_summary['adjusted_end_percent'] = (
        probability_summary['adjusted_end_probability'] * 100
    )
    display(
        probability_summary[
            [
                'population', 'work_mode', 'elo_reference',
                'adjusted_start_percent', 'adjusted_end_percent',
                'adjusted_change_points',
            ]
        ].round(2)
    )
    display(fixed_effects.round(4))
    display(variance_components.round(6))

if not fit_summary.empty:
    display(
        fit_summary[
            [
                'model_specification', 'iterations', 'elo_reference',
                'playlist_log_odds_slope', 'zpdes_log_odds_slope', 'error',
            ]
        ]
    )

,population,work_mode,elo_reference,adjusted_start_percent,adjusted_end_percent,adjusted_change_points
0,combined,playlist,1518.94,70.23,77.08,6.86
1,combined,zpdes,1518.94,49.37,84.14,34.77


,population,term,estimate,std_error,z_value,p_value,ci_low,ci_high,odds_ratio
0,combined,Intercept,0.8956,NaN,NaN,NaN,NaN,NaN,2.4487
1,combined,zpdes,-0.2140,NaN,NaN,NaN,NaN,NaN,0.8073
2,combined,centered_position,1.2309,0.0046,269.7608,0.0,1.2220,1.2399,3.4244
3,combined,zpdes_x_centered_position,1.3389,0.0099,134.9704,0.0,1.3194,1.3583,3.8147
4,combined,exercise_elo_centered_100,-0.6230,0.0010,-649.6225,0.0,-0.6249,-0.6212,0.5363


,population,group,variance
0,combined,classroom_id,0.649731
1,combined,student_id,0.619741
2,combined,sequence_id,0.880599
3,combined,sequence_id_rand_coef_centered_position,0.515860


,model_specification,iterations,elo_reference,playlist_log_odds_slope,zpdes_log_odds_slope,error
0,"Bernoulli-logit: centered work_mode * centered_position + exercise Elo; random intercepts for classroom, student, and sequence; independ...",31,1518.942585,0.35493,1.69379,None


## 8. Interpretation limits

- A positive adjusted slope is consistent with improving success at comparable Elo difficulty; it is not definitive evidence of causal learning.
- Random student effects account for repeated measurements and baseline heterogeneity but require the usual random-effects assumptions. Because the `combined` population includes students observed in only one mode, the work-mode contrast combines within-student and between-student information.
- Matching the half–half population does not make the numerical changes identical: this model estimates full start-to-end change on a logit scale, adjusts for exercise Elo, and weights attempt rows, whereas the half–half model compares two sequence-level half averages.
- The linear normalized-position term estimates one average start-to-end trend. Inspect the binned plot for strong curvature; a spline model is preferable if the trajectory is clearly nonlinear.
- Eligible sequences are conditioned on reaching at least four unique first-attempt exercises, so results do not generalize automatically to shorter or abandoned sequences.
- Sequence random intercepts and slopes model residual sequence heterogeneity. They do not add activity or module adjustment, because activity selection is treated as part of the work-mode system.
- If `slope_comparison_reportable` is true but `probability_levels_reportable` is false, the interaction coefficient and its interval can be reported, while adjusted start/end percentages must be labeled model-implied point estimates without complete baseline-level uncertainty.

## 9. Save outputs

In [8]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
attempt_summary.to_csv(OUTPUT_DIR / 'attempt_summary.csv', index=False, encoding='utf-8-sig')
sequence_summary.to_csv(OUTPUT_DIR / 'sequence_summary.csv', index=False, encoding='utf-8-sig')
elo_coverage.to_csv(OUTPUT_DIR / 'elo_coverage.csv', index=False, encoding='utf-8-sig')
binned_trajectory.to_csv(
    OUTPUT_DIR / 'observed_binned_trajectory.csv', index=False, encoding='utf-8-sig'
)
trajectory_figure.write_html(
    OUTPUT_DIR / 'observed_sequence_trajectory.html', include_plotlyjs='cdn'
)
if not fit_summary.empty:
    fit_summary.to_csv(OUTPUT_DIR / 'model_summary.csv', index=False, encoding='utf-8-sig')
if not teacher_summary.empty:
    teacher_summary.to_csv(
        OUTPUT_DIR / 'teacher_summary.csv', index=False, encoding='utf-8-sig'
    )
if not fixed_effects.empty:
    fixed_effects.to_csv(OUTPUT_DIR / 'fixed_effects.csv', index=False, encoding='utf-8-sig')
if not variance_components.empty:
    variance_components.to_csv(
        OUTPUT_DIR / 'variance_components.csv', index=False, encoding='utf-8-sig'
    )
if not adjusted_changes.empty:
    adjusted_changes.to_csv(
        OUTPUT_DIR / 'adjusted_probability_changes.csv',
        index=False,
        encoding='utf-8-sig',
    )

print(f'Saved outputs to: {OUTPUT_DIR}')

Saved outputs to: C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\artifacts\work_mode_sequence_slope_combined_notebook
